# Module 4 — Feature Selection

**By the end of this notebook, you will be able to:**
- Explain why adding every available column to a linear model can make it *worse*, not better
- Use univariate selection (`SelectKBest`) and recursive selection (`RFE`) to pick a smaller, effective feature set
- Explain why the two methods can disagree, and what that tells you about redundant features

**Context:** Every activity so far has used the same 8 hand-picked satellite columns (`DEFAULT_COLUMNS`). The raw data actually has 70 satellite columns — the 8 you have been using, plus secondary measurements (viewing angles, alternate retrievals, sensor geometry) that were deliberately set aside until now. This module uses all 70, and lets an algorithm choose instead of a human.

## Rebuild the cleaned dataset, on the full column set

**Exercise:** Same sequence as Module 2 (`restrict_to_scope` → `columns_above_missing_threshold` → `drop_columns` → `fill_missing_by_city` → `add_temporal_features`), but scoped to `SATELLITE_COLUMNS` below instead of the 8-column `DEFAULT_COLUMNS` — everything else is identical.

In [1]:
# TODO: rebuild enriched on the full satellite column set, reusing the
# Module 2 functions you already completed
from pathlib import Path
import sys

# Make the project's src/ package importable even if Jupyter starts in another folder/kernel.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "air_quality").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break

import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np

from air_quality import data, evaluation, features, selection
from air_quality.workflows import DEFAULT_COLUMNS

ALL_CITIES = ["Kampala", "Nairobi", "Lagos", "Bujumbura"]

train_df, _ = data.load_datasets()

ALL_COLUMNS = train_df.columns.tolist()

scoped = data.restrict_to_scope(train_df, ALL_CITIES, ALL_COLUMNS)

columns_to_drop = data.columns_above_missing_threshold(scoped, threshold=0.7)
cleaned = data.drop_columns(scoped, columns_to_drop)

fillable_columns = [col for col in cleaned.columns if col not in ("city", "date")]
cleaned = data.fill_missing_by_city(cleaned, fillable_columns)

enriched = features.add_temporal_features(cleaned)
feature_cols = features.feature_columns(enriched)

print("Nb Feature columns:", len(feature_cols))

enriched

Nb Feature columns: 66


,id,site_id,site_latitude,site_longitude,city,country,date,hour,sulphurdioxide_so2_column_number_density,sulphurdioxide_so2_column_number_density_amf,...,cloud_cloud_base_pressure,cloud_cloud_base_height,cloud_cloud_optical_depth,cloud_surface_albedo,cloud_sensor_azimuth_angle,cloud_sensor_zenith_angle,cloud_solar_azimuth_angle,cloud_solar_zenith_angle,pm2_5,dayofweek
7946,id_kckponblua,64ef31227771f10013f8b066,-3.380650,29.383830,Bujumbura,Burundi,2023-11-04,11,0.000053,0.834035,...,62186.304687,4134.122559,7.747695,0.334881,-100.412849,32.509655,-113.255730,33.186287,27.504000,5
7947,id_gshykfsukc,64ef31227771f10013f8b066,-3.380650,29.383830,Bujumbura,Burundi,2023-11-08,12,0.000053,0.834035,...,62186.304687,4134.122559,7.747695,0.334881,-100.412849,32.509655,-113.255730,33.186287,17.272750,2
7948,id_aqxf9kcpqu,64ef31227771f10013f8b066,-3.380650,29.383830,Bujumbura,Burundi,2023-11-09,11,0.000053,0.834035,...,27750.964844,10304.898437,8.031931,0.345833,-99.661995,41.890026,-114.965317,35.046368,6.688182,3
7949,id_wpfc6p151m,64ef31227771f10013f8b066,-3.380650,29.383830,Bujumbura,Burundi,2023-11-10,11,0.000053,0.834035,...,27750.964844,10304.898437,8.031931,0.345833,-99.661995,41.890026,-114.965317,35.046368,15.051000,4
7950,id_8z4vuea2wl,64ef31227771f10013f8b066,-3.380650,29.383830,Bujumbura,Burundi,2023-11-15,11,0.000053,0.834035,...,76482.046875,2458.005371,12.686726,0.317818,-101.260002,21.437737,-119.750206,32.664028,5.863500,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5813,id_tn51gcoz4a,644841fe1319140029aced6e,-1.283724,36.810364,Nairobi,Kenya,2024-01-27,10,0.000199,0.764215,...,76573.898438,2458.536865,4.868147,0.187310,73.323975,48.218334,-144.969040,21.264854,13.334286,5
7284,id_msdjbr87s4,644fa58ae5001000290df1e2,-1.286301,36.830761,Nairobi,Kenya,2024-01-27,10,-0.000128,0.729167,...,60248.230469,4453.724121,2.660630,0.219031,73.353371,47.893490,-144.867599,21.321957,21.272407,5
7611,id_d66ohje4sb,6246f308db9c64001e483ffe,-1.235120,36.812150,Nairobi,Kenya,2024-01-27,10,0.000299,0.767461,...,55880.978068,5131.594506,2.786974,0.231179,73.333183,48.052872,-145.004868,21.329876,9.172387,5
7928,id_eonffvuke4,62fe1f3d365de1001e86eb55,-1.224292,36.805005,Nairobi,Kenya,2024-01-27,10,0.000184,0.769604,...,55880.978068,5131.594506,2.786974,0.231179,73.333183,48.052872,-145.004868,21.329876,5.685071,5


## What if you just use all of them?

After Module 2's cleaning (drop-then-fill on this wider column set), you have 66 candidate features now, instead of 8. Try the obvious thing first: fit `LinearRegression` on all of them, and look at the fitted coefficients.

In [2]:
# Given: fit on every candidate feature, look at the largest coefficients
model = LinearRegression()
model.fit(enriched[feature_cols], enriched["pm2_5"])

coefficients = pd.Series(model.coef_, index=feature_cols).sort_values(key=np.abs, ascending=False)
print("pm2_5 range in this data:", enriched["pm2_5"].min(), "-", enriched["pm2_5"].max())
coefficients.head(10)

pm2_5 range in this data: 1.2000000000000002 - 456.1933333333334


nitrogendioxide_stratospheric_no2_column_number_density   -358824.586019
nitrogendioxide_tropospheric_no2_column_number_density      55046.748373
nitrogendioxide_no2_column_number_density                   53317.524281
nitrogendioxide_no2_slant_column_number_density            -36447.526577
formaldehyde_hcho_slant_column_number_density                6226.433120
sulphurdioxide_so2_slant_column_number_density              -4606.213303
sulphurdioxide_so2_column_number_density_15km                4218.138143
sulphurdioxide_so2_column_number_density                     3790.343933
formaldehyde_tropospheric_hcho_column_number_density        -2022.044345
ozone_o3_column_number_density                              -1520.010502
dtype: float64

## Do any of these columns say almost the same thing?

Before jumping to conclusions about *why* the coefficient above is so large, check directly: compute the correlation of every candidate feature against every **other** candidate feature (not against `pm2_5`), and look at the strongest pairs.

Two choices worth explaining before you do:
- **Pearson only, not Spearman.** Module 1 used both, because the question there was "is there a relationship at all, linear or not." Here the question is narrower: does a *linear* model's coefficient estimation get confused — a fact about the design matrix, specific to linear (Pearson) correlation.
- **No p-value.** With ~8000 rows, even a trivial correlation (r=0.1) has a p-value around 1e-16 — large samples make p-values stop being informative about anything except sample size. See [Lin, Lucas & Shmueli (2013), *Too Big to Fail: Large Samples and the p-Value Problem*](https://pubsonline.informs.org/doi/10.1287/isre.2013.0480). Only the size of `r` tells you anything useful here.

Also, unlike Module 1, no `dropna` gymnastics are needed: `enriched` was already fully cleaned two cells ago (0 missing values), so `DataFrame.corr()` can use every row for every pair.

In [3]:
# Given: correlation of every candidate feature against every other one
corr_matrix = enriched[feature_cols].corr().abs()
upper_triangle = np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
top_pairs = corr_matrix.where(upper_triangle).stack().sort_values(ascending=False)
top_pairs.head(10).round(4)

uvaerosolindex_solar_azimuth_angle   ozone_solar_azimuth_angle              0.9998
cloud_cloud_top_height               cloud_cloud_base_height                0.9995
cloud_cloud_top_pressure             cloud_cloud_base_pressure              0.9985
uvaerosolindex_solar_zenith_angle    ozone_solar_zenith_angle               0.9967
uvaerosolindex_sensor_azimuth_angle  ozone_sensor_azimuth_angle             0.9936
sulphurdioxide_solar_azimuth_angle   nitrogendioxide_solar_azimuth_angle    0.9928
uvaerosolindex_sensor_zenith_angle   ozone_sensor_zenith_angle              0.9907
nitrogendioxide_solar_azimuth_angle  formaldehyde_solar_azimuth_angle       0.9904
carbonmonoxide_solar_azimuth_angle   formaldehyde_solar_azimuth_angle       0.9903
                                     nitrogendioxide_solar_azimuth_angle    0.9897
dtype: float64

**Dozens of pairs above 0.9.** The strongest ones are not even the NO2 family: `uvaerosolindex_solar_azimuth_angle` and `ozone_solar_azimuth_angle` correlate at essentially **1.00**, and `cloud_cloud_top_height`/`cloud_cloud_base_height` at **0.999** — the same satellite-geometry or cloud-layer measurement, copied into different products' metadata. `nitrogendioxide_no2_column_number_density` and its `tropospheric` variant, close to 0.97, are only one example among many.

This is the same *family* of problem as `site_latitude`/`site_longitude` in Session 2 — a linear model producing a huge, unstable coefficient — but a different cause. Back then, one feature barely varied within a city and jumped to a very different value in another. Here, many features carry almost the exact same information: when two columns say almost the same thing, a linear model cannot uniquely decide how much credit to assign to each, so small changes in the data can swing the split between them wildly. This is called **multicollinearity**, and it is exactly what produced the coefficient you saw above.

See what this actually does to generalization: evaluate this same feature set with `evaluate_group_cv` (Module 3).

In [4]:
# Given: expect this to look much worse than anything you have seen so far
result_all = evaluation.evaluate_group_cv(LinearRegression(), enriched, feature_cols, groups_col="city")
{k: v for k, v in result_all.items() if k != "folds"}

{'rmse_mean': np.float64(3995.0284545970544),
 'rmse_std': np.float64(6863.371508046034),
 'mae_mean': np.float64(282.4085336483932),
 'mae_std': np.float64(461.36156493150634),
 'r2_mean': np.float64(-306446.8223114534),
 'r2_std': np.float64(530781.5584529691)}

Unstable in-sample coefficients become catastrophic out-of-sample: trained on three cities, the model's huge, barely-determined coefficients get multiplied by a fourth city's slightly different values, and the error explodes far beyond anything you have seen in this course. More features is not automatically better.

[`scikit-learn`'s feature selection guide](https://scikit-learn.org/stable/modules/feature_selection.html) documents two families of methods to cut this down: **univariate** (score each feature against the target, independently) and **recursive** (fit repeatedly, dropping the least useful feature each time).

## Univariate selection: `SelectKBest`

**Exercise:** Complete `select_k_best_features` in `src/air_quality/selection.py` (new file, already scaffolded) — its docstring and `tests/test_selection.py` specify exactly what it should do. Run `uv run pytest tests/test_selection.py -v` until it passes, then apply it here with `k=8`, the same number of columns Session 2 used.

In [5]:
# TODO: apply select_k_best_features once you have implemented it
skb_selected = selection.select_k_best_features(enriched, feature_cols, target_col="pm2_5", k=8)
print("Nb feature columns after selection:", len(skb_selected))
print("Selected features:", skb_selected)

Nb feature columns after selection: 8
Selected features: ['sulphurdioxide_solar_azimuth_angle', 'carbonmonoxide_co_column_number_density', 'carbonmonoxide_solar_azimuth_angle', 'nitrogendioxide_no2_column_number_density', 'nitrogendioxide_tropospheric_no2_column_number_density', 'nitrogendioxide_no2_slant_column_number_density', 'uvaerosolindex_solar_azimuth_angle', 'ozone_solar_azimuth_angle']


**Look closely at the names.** Several of them end in `solar_azimuth_angle`, from *different* satellite products (sulphurdioxide, carbonmonoxide, ozone, uvaerosolindex) — exactly the kind of near-duplicate pair the correlation diagnostic above surfaced. Confirm it directly on this specific selection.

In [6]:
# Given: correlation among the SelectKBest picks themselves
enriched[skb_selected].corr().round(2)

,sulphurdioxide_solar_azimuth_angle,carbonmonoxide_co_column_number_density,carbonmonoxide_solar_azimuth_angle,nitrogendioxide_no2_column_number_density,nitrogendioxide_tropospheric_no2_column_number_density,nitrogendioxide_no2_slant_column_number_density,uvaerosolindex_solar_azimuth_angle,ozone_solar_azimuth_angle
sulphurdioxide_solar_azimuth_angle,1.00,-0.54,0.99,-0.20,-0.22,-0.14,0.98,0.98
carbonmonoxide_co_column_number_density,-0.54,1.00,-0.54,0.53,0.52,0.44,-0.53,-0.53
carbonmonoxide_solar_azimuth_angle,0.99,-0.54,1.00,-0.21,-0.23,-0.15,0.98,0.98
nitrogendioxide_no2_column_number_density,-0.20,0.53,-0.21,1.00,0.97,0.83,-0.22,-0.22
nitrogendioxide_tropospheric_no2_column_number_density,-0.22,0.52,-0.23,0.97,1.00,0.81,-0.23,-0.23
nitrogendioxide_no2_slant_column_number_density,-0.14,0.44,-0.15,0.83,0.81,1.00,-0.16,-0.16
uvaerosolindex_solar_azimuth_angle,0.98,-0.53,0.98,-0.22,-0.23,-0.16,1.00,1.00
ozone_solar_azimuth_angle,0.98,-0.53,0.98,-0.22,-0.23,-0.16,1.00,1.00


These four correlate at **1.00** with each other, same as before. Each one independently looks like a real predictor of `pm2_5` (it is a proxy for time of day), so `SelectKBest` happily keeps all four — spending half of its 8-column budget on one piece of information repeated four times. It has no way to know they are redundant: it never looks at more than one feature at a time.

## Recursive selection: `RFE`

**Exercise:** Complete `select_features_rfe` in the same file. Then apply it with `LinearRegression()` and `n_features_to_select=8`, and compare its picks to `SelectKBest`'s.

In [10]:
# TODO: apply select_features_rfe once you have implemented it
rfe_selected = selection.select_features_rfe(model=LinearRegression(), df=enriched, feature_cols=feature_cols, target_col="pm2_5", n_features_to_select=8)
print("Nb feature columns after selection:", len(rfe_selected))
print("Selected features:", rfe_selected)

Nb feature columns after selection: 8
Selected features: ['sulphurdioxide_so2_column_number_density', 'sulphurdioxide_so2_slant_column_number_density', 'carbonmonoxide_co_column_number_density', 'nitrogendioxide_no2_column_number_density', 'nitrogendioxide_tropospheric_no2_column_number_density', 'nitrogendioxide_stratospheric_no2_column_number_density', 'nitrogendioxide_no2_slant_column_number_density', 'formaldehyde_tropospheric_hcho_column_number_density']


In [8]:
# Given: what do the two methods agree and disagree on?
print("kept by both:", set(skb_selected) & set(rfe_selected))
print("SelectKBest only:", set(skb_selected) - set(rfe_selected))
print("RFE only:", set(rfe_selected) - set(skb_selected))

kept by both: {'nitrogendioxide_no2_column_number_density', 'nitrogendioxide_no2_slant_column_number_density', 'carbonmonoxide_co_column_number_density', 'nitrogendioxide_tropospheric_no2_column_number_density'}
SelectKBest only: {'sulphurdioxide_solar_azimuth_angle', 'ozone_solar_azimuth_angle', 'uvaerosolindex_solar_azimuth_angle', 'carbonmonoxide_solar_azimuth_angle'}
RFE only: {'nitrogendioxide_stratospheric_no2_column_number_density', 'formaldehyde_tropospheric_hcho_column_number_density', 'sulphurdioxide_so2_column_number_density', 'sulphurdioxide_so2_slant_column_number_density'}


RFE fits with all remaining features still in the model at each step: once one `solar_azimuth_angle` column is in, its near-duplicates add almost nothing more, so they tend to be the ones eliminated first — RFE's picks lean toward genuinely different pollutant measurements instead.

## Does it actually matter? Measure it

Compare all three feature sets with `evaluate_group_cv`: every candidate feature, `SelectKBest`'s 8, and RFE's 8.

In [11]:
# Given: same evaluation, three feature sets
for name, cols in [
    ("all 66 features", feature_cols),
    ("SelectKBest (k=8)", skb_selected),
    ("RFE (k=8)", rfe_selected),
]:
    result = evaluation.evaluate_group_cv(LinearRegression(), enriched, cols, groups_col="city")
    print(f"{name}: rmse_mean={result['rmse_mean']:.2f}  r2_mean={result['r2_mean']:.3f}")

all 66 features: rmse_mean=3995.03  r2_mean=-306446.822
SelectKBest (k=8): rmse_mean=32.84  r2_mean=-1.187
RFE (k=8): rmse_mean=27.63  r2_mean=0.068


**Continue on the site:** [Module 4 — Feature Selection](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_4/)